# Applied AI in Chemical and Process Engineering
## Week 3/4: Machine Learning Pipeline - Hands-on

---

### Learning Objectives
1. **Chemical Process Data Ingestion**
2. **Process Exploratory Data Analysis (EDA)**
3. **Data Cleaning**
4. **Preprocessing**
5. **ML Model building**
6. **Model Evaluation & Residual Diagnostics**
7. **Explainable AI (SHAP)**
8. **Process Optimization**


## Part 1 · Chemical Process Data Ingestion

### 1. Industrial Problem Formulation
In industrial bioethanol manufacturing, cane molasses (rich in fermentable sugars) is fermented in batch bioreactors using *Saccharomyces cerevisiae* (yeast).

Plant operators control four primary operating levers:
1. **Temperature ($T$, °C)**: Broth temperature regulated via cooling jackets to balance metabolic rate against thermal enzyme stress.
2. **Broth Acidity ($\text{pH}$)**: Maintained via buffer addition to support active yeast enzymatic pathways.
3. **Yeast Inoculum ($X_{\text{yeast}}$, g/L)**: The concentration of active biocatalyst added at the start of the batch.
4. **Initial Feed Sugar ($S_0$, g/L)**: The starting substrate concentration loaded into the bioreactor.

**ML Objective**: Build a predictive regression model to forecast the **Residual (Unfermented) Sugar ($S_{residual}$, g/L)** remaining at the end of the batch run, enabling operators to minimize sugar loss and stabilize batch operations.

---

### 2. Systematic Process Variable Definitions

| Variable Symbol | Column Name in Dataset | Engineering Description | Typical Range / Units | Role in ML Pipeline |
| :---: | :--- | :--- | :---: | :--- |
| **$T$** | `Temperature` | Fermentation broth temperature | $25.0 - 38.0\text{ }^\circ\text{C}$ | Input Feature ($X_1$) |
| **$\text{pH}$** | `pH` | Broth acidity/alkalinity level | $4.0 - 5.5$ | Input Feature ($X_2$) |
| **$X_{\text{yeast}}$** | `Yeast_Concentration` | Inoculum biomass catalyst concentration | $1.0 - 5.0\text{ g/L}$ | Input Feature ($X_3$) |
| **$S_0$** | `Sugar_Concentration` | Initial feed sugar (substrate) concentration | $10.0 - 20.0\text{ g/L}$ | Input Feature ($X_4$) |
| **$S_{residual}$** | `Residual_Sugar` | Unfermented sugar remaining after batch reaction | $0.8 - 11.0\text{ g/L}$ | **Primary Target Output ($y$)** |


### Vibe-Coding Prompt 1 · Environment Setup & Data Ingestion

**Prompt to LLM:**

Role: Chemical Process Data Scientist
Context: Analyzing industrial molasses fermentation batch data for bioethanol production.
Task: Import numpy, pandas, matplotlib, seaborn, scikit-learn, xgboost, optuna, shap, and scipy. Install if any missing library


Load the dataset from 'https://raw.githubusercontent.com/dissabnd/Applied-AI-CPE-UG-2026/refs/heads/main/data/Ethanol_Molasses_Dataset.csv'

Display the shape, column data types, and the first 5 rows.

## Part 2 · Process Exploratory Data Analysis (EDA)

Before training machine learning models, chemical engineers examine variable distributions to identify operating envelopes, skewness, and multi-variable correlations.

<div align="center">
  <table>
    <tr>
      <td align="center"><b>Box Plot Anatomy (Outliers & Quartiles)</b><br><img src="./assets/boxplot.png" width="380"></td>
      <td align="center"><b>Violin Plot Anatomy (Probability Density)</b><br><img src="./assets/violinplot.png" width="380"></td>
    </tr>
  </table>
</div>

### Vibe-Coding Prompt 2 · Data Visualization (before cleaning)

**Prompt to LLM:**

Role: Biochemical Engineer & Data Scientist
Context: Dataset contains 4 operating features ['Temperature', 'pH', 'Yeast_Concentration', 'Sugar_Concentration'] and target 'Residual_Sugar'.
Task: 
1. Create a 2x3 grid of subplots displaying the distribution (Histogram with KDE) for each variable, plus a Pearson correlation heatmap in the 6th subplot.
2. Create a 1x4 grid of scatter plots (regplot) displaying the target Residual_Sugar on the y-axis against each input variable on the x-axis to visually inspect trends and non-linearities.
Constraints: Include proper physical unit labels (°C, g/L), distinct colors, scatter markers with alpha transparency, and bold titles.

## Part 3 · Data Cleaning (Deduplication & Sensor Outlier Filtering)

Real industrial plant historians (SCADA/DCS) frequently suffer from data quality anomalies before machine learning models can be trained:
1. **Duplicate Telemetry Records**: Caused by network polling re-tries, sensor heartbeat repeats, or database sync duplicates.
2. **Gross Physical Outliers**: Caused by thermocouple detachment, pH probe fouling, or transmission corruption (e.g. $T = 82^\circ\text{C}$ in a yeast fermenter where cells perish above $40^\circ\text{C}$, or unphysical acidic/alkaline readings $\text{pH} < 3.0$ / $\text{pH} > 7.0$).

> [!NOTE]
> **Engineering Rule**: Physical outliers that violate basic thermodynamic possibility ($T > 40^\circ\text{C}$, $\text{pH} < 3.0$ or $\text{pH} > 7.0$) must be identified and removed prior to model training. After cleaning, we re-plot the entire visualization suite to visually inspect the clean distributions and process response surfaces.

### Vibe-Coding Prompt 3 · Data Cleaning & Post-Cleaning Visual Verification

**Prompt to LLM:**

Role: Industrial Data Quality Engineer
Context: Telemetry DataFrame `df` with process columns `['Temperature', 'pH', 'Yeast_Concentration', 'Sugar_Concentration', 'Residual_Sugar']`.
Task:

1. Identify and remove duplicate rows using `df.drop_duplicates()`.
2. Identify and filter out unphysical sensor failure outliers (valid domain bounds: $20.0 \le \text{Temperature} \le 40.0^\circ\text{C}$ and $3.0 \le \text{pH} \le 7.0$).
3. Print a formatted data cleaning report showing initial raw records, duplicate rows removed, sensor outliers removed, and final clean dataset dimensions.
4. Assign the cleaned result back to `df`.
5. Re-plot all process visualizations on the cleaned data:
   - A 2x3 grid displaying clean distributions (Histogram with KDE) for each variable and a clean Pearson correlation matrix.
   - A 1x4 grid of scatter plots (`regplot`) displaying clean Residual_Sugar ($y$) against each operating lever ($x$) with trendlines to clearly reveal the true underlying bioprocess curves.

## Part 4 · Preprocessing

<div align="center">
  <img src="./assets/datasplit.jpg" alt="Train/Test Split Strategy" width="600">
</div>

> [!IMPORTANT]
> **The Golden Rule of ML Engineering**: Always perform `train_test_split` **BEFORE** any imputation, scaling, or transformation!
> If you compute median imputation or standard scaling on the *entire dataset*, future test set information leaks into your training pipeline, giving deceptively optimistic validation metrics that fail in the factory.

### Vibe-Coding Prompt 4 ·  Train/Test Split & Median Imputation

**Prompt to LLM:**

Role: Machine Learning Engineer
Context: Cleaned dataset has features `['Temperature', 'pH', 'Yeast_Concentration', 'Sugar_Concentration']` and target `'Residual_Sugar'`.
Task:
1. Split data into 80% train and 20% test using `train_test_split(random_state=42)`.
2. Impute missing values with `SimpleImputer(strategy='median')` strictly fitted on `X_train`, then transform `X_test`.
3. Wrap back into Pandas DataFrames with original column names.
4. Verify missing values in `X_train` and `X_test` are zero.

## Part 5 · ML Model Building (Baseline Training & Hyperparameter Tuning)

### 5.1 Baseline Gradient Boosted Regression (XGBoost)

<div align="center">
  <img src="./assets/xgboost.png" alt="XGBoost Gradient Boosting Concept" width="600">
</div>

In industrial tabular regression, **Extreme Gradient Boosting (XGBoost)** builds an ensemble of shallow decision trees sequentially, where each new tree fits the residual errors of prior trees.

### Evaluation Metrics:
* **Coefficient of Determination ($R^2$)**: Proportion of variance explained ($1.0$ is perfect, $<0.0$ is worse than predicting mean).
* **Root Mean Squared Error (RMSE)**: Penalizes large deviations in physical units ($	ext{g/L}$).
* **Mean Absolute Error (MAE)**: Average absolute prediction deviation ($	ext{g/L}$).

### Vibe-Coding Prompt 5.1 · Standard XGBoost Baseline Training

**Prompt to LLM:**

Role: Applied AI Specialist
Task:
1. Train a standard baseline model: `XGBRegressor(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)` on `X_train_imp`.
2. Predict on `X_test_imp` and calculate $R^2$, RMSE, and MAE against `y_test`.
3. Print a formatted baseline performance table.

### 5.2 Modern Bayesian Hyperparameter Optimization with Optuna

<div align="center">
  <img src="./assets/optuna.png" alt="Hyperparameter Search Strategy" width="1000">
</div>

Instead of brute-force `GridSearchCV` (which evaluates redundant points on a static grid), modern industrial ML uses **Optuna** (Bayesian Tree-structured Parzen Estimator).
* **Efficiency**: Finds superior hyperparameters in 15–20 trials instead of hundreds of grid iterations.
* **Pruning & History**: Automatically tracks parameter importance and convergence trajectories.

### Vibe-Coding Prompt 5.2 · Optuna Bayesian Hyperparameter Optimization

**Prompt to LLM:**

Role: AI Performance Engineer
Task:
1. Define an Optuna objective function optimizing `XGBRegressor` hyperparameters:
   - `n_estimators` (50 to 250)
   - `max_depth` (3 to 6)
   - `learning_rate` (0.01 to 0.2 log scale)
   - `subsample` (0.6 to 1.0)
2. Use 5-fold cross-validation (`KFold(n_splits=5, shuffle=True, random_state=42)`) with `neg_root_mean_squared_error`.
3. Run study for 20 trials with fixed random seed and retrain `best_model` on full `X_train_imp`.

## Part 6 · Model Evaluation & Residual Diagnostics

<div align="center">
  <img src="./assets/residualplot.png" alt="Residual Diagnostics Concept" width="650">
</div>

### Model Diagnostic Principles:
1. **Parity Plot (Actual vs. Predicted)**: Data points should align tightly along the $y = x$ 45° diagonal line.
2. **Residual Plot ($e_i = y_i - \hat{y}_i$)**: Residuals must be randomly scattered around zero with no systematic curvature (homoscedasticity).

### Vibe-Coding Prompt 6 · Model Evaluation & Parity Diagnostics

**Prompt to LLM:**

Role: Chemical Process Quality & Validation Engineer
Context: Evaluating test predictions `y_test_pred` against experimental measurements `y_test`.
Task:
1. Calculate evaluation metrics: $R^2$ (`r2_score`), RMSE (`np.sqrt(mean_squared_error)`), and MAE (`mean_absolute_error`) on the test set.
2. Create a 1x2 subplot figure (figsize=(13, 5)) for model validation.
3. Subplot 1 (Parity Plot): Scatter actual `y_test` vs. predicted `y_test_pred` with a red dashed 45° reference line ($y = x$). Display $R^2$ and RMSE values directly inside the plot via an annotated text box.
4. Subplot 2 (Residual Plot): Compute residuals ($e = y_{test} - \hat{y}_{test}$) and scatter predicted values vs. residuals with a horizontal reference line at $e = 0$.
5. Print the numerical $R^2$, RMSE, and MAE values to the console.

## Part 7 · Explainable AI (SHAP Kinetic Discovery)

<div align="center">
  <table>
    <tr>
      <td align="center"><b>Explainable AI (XAI) Framework</b><br><img src="./assets/xai.png" width="420"></td>
      <td align="center"><b>SHAP Feature Attribution Concept</b><br><img src="./assets/shap.png" width="420"></td>
    </tr>
  </table>
</div>

In Chemical Engineering, **Explainable AI (XAI)** is an essential model validation tool:
1. **Global Feature Hierarchy**: Which operating levers exert the strongest control over residual sugar?
2. **Kinetic Shape Discovery**: Does SHAP reveal the expected biological enzyme optimum for Temperature ($30\text{--}33^\circ\text{C}$) and pH ($4.5\text{--}5.0$)? Bubble curves should demonstrate thermal deactivation beyond optimum.

### Vibe-Coding Prompt 7 · SHAP Global Feature Importance & Kinetic Discovery

**Prompt to LLM:**

Role: Bioprocess Domain Expert & Explainable AI Researcher
Context: Using TreeSHAP on `best_model` with `X_train_imp` to interpret kinetic mechanisms.
Task:
1. Initialize `shap.TreeExplainer` on `best_model` and compute SHAP values for `X_train_imp`.
2. Generate a SHAP Beeswarm summary plot to visualize feature ranking and impact directions on Residual Sugar.
3. Plot 1x2 SHAP kinetic dependence scatter plots for 'Temperature' and 'pH', adding vertical dashed reference lines at biological enzyme optima (32.0°C and pH 4.8).

## Part 8 · Process Optimization (Finding Optimal Operating Conditions)

<div align="center">
  <img src="./assets/optimization.jpg" alt="Optimization Search Strategy" width="600">
</div>

Now that we have trained and validated our surrogate ML model `best_model`, we can solve the **bioprocess operating optimization** problem:
Finding the optimal operating setpoints $(T^*, \text{pH}^*, X^*, S_0^*)$ that **minimize unreacted Residual Sugar** ($\\hat{y}$).

### Why Bayesian Optimization for Process Optimization?
Because tree-based models (XGBoost) produce step-wise, non-smooth surfaces where gradient-based optimizers (like gradient descent or L-BFGS) get trapped in flat plateaus, **Bayesian Optimization (Optuna)** is the ideal black-box global optimizer to explore the parameter space and discover the true operating optimum.

### Optimization Problem Formulation:
$$\min_{T, \text{pH}, X, S_0} \hat{S}_{residual}(T, \text{pH}, X, S_0)$$
$$\text{Subject to: } 25.0 \le T \le 38.0^\circ\text{C}, \quad 4.0 \le \text{pH} \le 5.5, \quad 1.0 \le X \le 5.0\text{ g/L}, \quad 12.0 \le S_0 \le 18.0\text{ g/L}$$

### Vibe-Coding Prompt 8 · Bayesian Bioprocess Operating Optimization

**Prompt to LLM:**

Role: Industrial Optimization Engineer
Context: Using the trained XGBoost surrogate model `best_model` to discover the optimal plant operating conditions.
Task:
1. Define an Optuna objective function `process_objective(trial)` that suggests:
   - `Temperature` in [25.0, 38.0] °C
   - `pH` in [4.0, 5.5]
   - `Yeast_Concentration` in [1.0, 5.0] g/L
   - `Sugar_Concentration` in [12.0, 18.0] g/L
   and returns the predicted `Residual_Sugar` from `best_model`.
2. Create an Optuna study with `direction='minimize'` (using `TPESampler(seed=42)`).
3. Optimize for 100 trials and print a formatted report of the optimal operating levers and the minimum predicted residual sugar.
4. Verify that the optimal conditions align with the enzyme peaks identified in the SHAP analysis.